# 1. Project Setup and Initial Data Inspection

## Objective
Load the Zomato restaurant dataset and understand its structure before performing data cleaning or exploratory data analysis.

## What we will check
- Dataset dimensions (rows and columns)
- Column names and data types
- Sample restaurant records
- Missing values and duplicate records
- Whether review text and restaurant ratings are available

## Best practice
We will preserve the original dataset and perform all subsequent cleaning on a separate working copy. This helps maintain reproducibility and prevents accidental modification of the source data.

## Step 1: Load the Dataset

**Objective:** Load the Zomato CSV file into a Pandas DataFrame so we can explore and analyze the data.

In [ ]:
import pandas as pd

df = pd.read_csv("data/zomato.csv")

print("Dataset loaded successfully!")

Dataset loaded successfully!


## Step 2: Check Dataset Dimensions

**Objective:** Identify how many rows and columns are present in the Zomato dataset.

This helps us understand the size of our dataset before starting data cleaning and exploratory data analysis.

In [15]:
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]:,}")

Number of rows: 51,717
Number of columns: 17


## 2.1 Inspect Dataset Columns

Display the column names to identify the available restaurant attributes, particularly restaurant names, ratings, and customer reviews. These fields will guide the design of our content-based recommendation system.

In [ ]:
print("Dataset columns:")
print(df.columns.tolist())

Dataset columns:
['url', 'address', 'name', 'online_order', 'book_table', 'rate', 'votes', 'phone', 'location', 'rest_type', 'dish_liked', 'cuisines', 'approx_cost(for two people)', 'reviews_list', 'menu_item', 'listed_in(type)', 'listed_in(city)']


## 2.2 Inspect Data Types and Missing Values

Examine each column's data type and count of non-null values. This helps identify missing data and determine which columns require cleaning before NLP preprocessing and recommendation modeling.

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51717 entries, 0 to 51716
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   url                          51717 non-null  str  
 1   address                      51717 non-null  str  
 2   name                         51717 non-null  str  
 3   online_order                 51717 non-null  str  
 4   book_table                   51717 non-null  str  
 5   rate                         43942 non-null  str  
 6   votes                        51717 non-null  int64
 7   phone                        50509 non-null  str  
 8   location                     51696 non-null  str  
 9   rest_type                    51490 non-null  str  
 10  dish_liked                   23639 non-null  str  
 11  cuisines                     51672 non-null  str  
 12  approx_cost(for two people)  51371 non-null  str  
 13  reviews_list                 51717 non-null  str  
 14  m

## 2.3 Missing Value Analysis

Calculate the number and percentage of missing values in each column. Sort the results by the number of missing values so that the most affected columns are easy to identify.

We will use these findings to decide how to handle missing values without unnecessarily removing useful restaurant records.

In [25]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values(by="Missing Count", ascending=False)

display(missing_summary)

,Missing Count,Missing Percentage
dish_liked,28078,54.29
rate,7775,15.03
phone,1208,2.34
approx_cost(for two people),346,0.67
rest_type,227,0.44
cuisines,45,0.09
location,21,0.04


## 2.4 Explore Restaurant Ratings

Examine the most frequent values in the rating column, including missing values. This will help us understand the rating format and identify non-numeric entries before cleaning and converting ratings into numeric values.

In [26]:

display(
    df["rate"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("Rating Value")
    .reset_index(name="Count")
)


,Rating Value,Count
0,NaN,7775
1,NEW,2208
2,3.9/5,2098
3,3.8/5,2022
4,3.7/5,2011
5,3.9 /5,1874
6,3.8 /5,1851
7,3.7 /5,1810
8,3.6/5,1773
9,4.0/5,1609


## 2.5 Explore Customer Reviews

Inspect sample values from the reviews_list column to understand how customer reviews are stored. This will help determine how to extract and aggregate review text for each restaurant before applying NLP techniques.

In [27]:
# Display three sample restaurant names and their reviews
display(df[["name", "reviews_list"]].head(3))

,name,reviews_list
0,Jalsa,"[('Rated 4.0', 'RATED\n A beautiful place to ..."
1,Spice Elephant,"[('Rated 4.0', 'RATED\n Had been here for din..."
2,San Churro Cafe,"[('Rated 3.0', ""RATED\n Ambience is not that ..."


## 2.6 Inspect a Complete Review Entry

Display the complete reviews_list value for one restaurant. Examine the review format, including rating labels, review text, and any special characters. This will guide our text extraction and preprocessing strategy.

In [28]:
print(df.loc[0, "reviews_list"])

[('Rated 4.0', 'RATED\n  A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here again.'), ('Rated 4.0', 'RATED\n  I was here for dinner with my family on a weekday. The restaurant was completely empty. Ambience is good with some good old hindi music. Seating arrangement are good too. We ordered masala papad, panner and baby corn starters, lemon and corrionder soup, butter roti, olive and chilli paratha. Food was fresh and good, service is good too. Good for family hangout.\nCheers'), ('Rated 2.0', 'RATED\n  Its a restaurant near to Banashankari BDA. Me along with few of my office friends visited to have buffet but unfortunately they only provide veg buffet. On inquiring they said this place is mostly visi

## 2.7 Analyze Restaurant Name Uniqueness

Count the total records, unique restaurant names, and duplicated restaurant-name records. This helps us understand whether the dataset contains multiple entries for the same restaurant and informs our strategy for creating restaurant-level review profiles.

Restaurant names alone may not uniquely identify a physical outlet, so we will investigate duplicates before deciding how to group records.

In [29]:
print("Total records:", len(df))
print("Unique restaurant names:", df["name"].nunique())
print("Records with duplicated restaurant names:", df["name"].duplicated().sum())

Total records: 51717
Unique restaurant names: 8792
Records with duplicated restaurant names: 42925


## 2.8 Investigate Repeated Restaurant Names

Identify the most frequently occurring restaurant names and inspect their associated locations, ratings, and review records. This will help determine whether repeated names represent multiple outlets, duplicate listings, or distinct records that should be retained separately.

In [30]:
# Identify the 10 most frequently occurring restaurant names
top_restaurants = df["name"].value_counts().head(10).index

# Inspect their names, locations, ratings, and review counts
display(
    df.loc[
        df["name"].isin(top_restaurants),
        ["name", "location", "rate", "reviews_list"]
    ]
    .assign(
        Review_Text_Length=lambda data: data["reviews_list"].str.len()
    )
    .drop(columns="reviews_list")
    .sort_values("name")
    .head(30)
)

,name,location,rate,Review_Text_Length
12024,Baskin Robbins,Vasanth Nagar,3.1/5,2878
13542,Baskin Robbins,Electronic City,3.8/5,879
49148,Baskin Robbins,Bellandur,3.7 /5,473
3186,Baskin Robbins,Basavanagudi,3.7/5,1619
39011,Baskin Robbins,Brigade Road,3.7 /5,262
2742,Baskin Robbins,Jayanagar,3.6/5,467
7343,Baskin Robbins,Brookefield,3.9/5,611
45508,Baskin Robbins,Old Airport Road,3.7 /5,905
49567,Baskin Robbins,Sarjapur Road,NaN,507
7682,Baskin Robbins,Brookefield,3.9/5,611


## 2.9 Detect Duplicate Records

Check for completely identical rows and for repeated combinations of restaurant name, location, and review text. This helps distinguish potential duplicate listings from separate restaurant outlets.

We will identify duplicates first and will not remove any records until their impact on the recommendation system is understood.

In [31]:
print("Completely duplicated rows:", df.duplicated().sum())

print(
    "Duplicate records by restaurant name, location, and reviews:",
    df.duplicated(subset=["name", "location", "reviews_list"]).sum()
)

Completely duplicated rows: 0
Duplicate records by restaurant name, location, and reviews: 26600


## 2.10 Identify Unique Restaurant Outlets

Count unique combinations of restaurant name and location. This provides a more useful estimate of distinct restaurant outlets than counting names alone, because the same restaurant brand may operate across multiple locations.

This is an initial grouping strategy; we will validate it before creating the final recommendation profiles.

In [32]:
unique_outlets = df[["name", "location"]].drop_duplicates()

print("Unique restaurant names:", df["name"].nunique())
print("Unique restaurant outlets (name + location):", len(unique_outlets))
print("Records beyond unique outlet combinations:", len(df) - len(unique_outlets))

Unique restaurant names: 8792
Unique restaurant outlets (name + location): 12151
Records beyond unique outlet combinations: 39566


## 2.11 Analyze Records per Restaurant Outlet

Count the number of records associated with each restaurant name and location. Examine the distribution to understand how frequently outlets appear in the dataset.

This analysis will help identify repeated listings and guide our strategy for constructing a single review profile for each outlet.

In [33]:
records_per_outlet = (df.groupby(["name", "location"], dropna=False).size().rename("record_count"))

print("Outlet record-count distribution:")
display(records_per_outlet.describe().to_frame().T)

print("\nMost frequently occurring outlet combinations:")
display(
    records_per_outlet
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

Outlet record-count distribution:


,count,mean,std,min,25%,50%,75%,max
record_count,12151.0,4.256193,3.410459,1.0,2.0,3.0,5.0,34.0



Most frequently occurring outlet combinations:


,name,location,record_count
0,Hammered,Cunningham Road,34
1,Dolci Desserts,Cunningham Road,25
2,The Chocolate Heaven,Commercial Street,24
3,Tiger Trail - Ramada Hotel,Shivajinagar,23
4,ThrowBack - Pub & Dine,Langford Town,22
5,Hatti Punjab Di,Brigade Road,22
6,Berrylicious Cafe,MG Road,22
7,Stoner,Koramangala 5th Block,21
8,Fresh Pressery Cafe,Koramangala 5th Block,21
9,Cilantro,Koramangala 4th Block,21


## 3.1 Analyze Review Text Length

Measure the character length of each restaurant's reviews_list field. This helps identify unusually short or long entries and understand the amount of text available for NLP.

We will retain the original review data and use this analysis to guide subsequent preprocessing.

In [34]:
review_lengths = df["reviews_list"].str.len()

print("Review text length statistics:")
display(review_lengths.describe().to_frame().T)

print("\nFive shortest review entries:")
display(
    df.loc[
        review_lengths.nsmallest(5).index,
        ["name", "location", "reviews_list"]
    ]
    .assign(Review_Text_Length=lambda data: data["reviews_list"].str.len())
    .drop(columns="reviews_list")
)

Review text length statistics:


,count,mean,std,min,25%,50%,75%,max
reviews_list,51717.0,9518.05416,27654.75738,2.0,326.0,1170.0,4179.0,1284117.0



Five shortest review entries:


,name,location,Review_Text_Length
75,Om Sri Vinayaka Chats,Banashankari,2
91,Sunsadm,Banashankari,2
92,Annapooraneshwari Mess,Banashankari,2
95,Vi Ra's Bar and Restaurant,Banashankari,2
107,Coffee Shopee,Banashankari,2


## Inspect Empty and Very Short Review Entries

In this step, we will inspect the `reviews_list` column to identify empty lists, blank entries, and unusually short review data.

This check will help us understand how much usable review information is available before extracting and preprocessing customer reviews for the recommendation system.

We will preserve the original dataset and handle entries with missing or insufficient review text during data preparation.

In [35]:
print("Review entries with 2 characters or fewer:")
display(
    df.loc[
        df["reviews_list"].str.len() <= 2,
        ["name", "location", "reviews_list"]
    ].head(10)
)

print("\nExact empty-list entries:")
print((df["reviews_list"].str.strip() == "[]").sum())

print("\nBlank review entries:")
print((df["reviews_list"].str.strip() == "").sum())

print("\nTotal rows with review entries of 2 characters or fewer:")
print((df["reviews_list"].str.len() <= 2).sum())

Review entries with 2 characters or fewer:


,name,location,reviews_list
75,Om Sri Vinayaka Chats,Banashankari,[]
91,Sunsadm,Banashankari,[]
92,Annapooraneshwari Mess,Banashankari,[]
95,Vi Ra's Bar and Restaurant,Banashankari,[]
107,Coffee Shopee,Banashankari,[]
108,Naveen Kabab & Biriyani Mane,Banashankari,[]
109,Katriguppe Donne Biryani,Banashankari,[]
112,Just Bake,Banashankari,[]
113,Dharwad Line Bazaar Mishra Pedha,Banashankari,[]
115,Aarush's Food Plaza,Banashankari,[]



Exact empty-list entries:
7595

Blank review entries:
0

Total rows with review entries of 2 characters or fewer:
7595


## Inspect the Structure of Actual Customer Reviews

In this step, we will inspect a few non-empty `reviews_list` entries to understand their structure and identify how individual customer reviews and their rating labels are stored.

This inspection will guide us in safely extracting the review text for NLP preprocessing and building the restaurant recommendation model.

In [36]:
# Select rows that contain review data
sample_reviews = df.loc[
    df["reviews_list"].str.strip() != "[]",
    ["name", "location", "reviews_list"]
].head(3)

# Display the first 1,000 characters of each review entry
for index, row in sample_reviews.iterrows():
    print(f"Restaurant: {row['name']}")
    print(f"Location: {row['location']}")
    print(f"Review data preview:\n{row['reviews_list'][:1000]}")
    print("-" * 80)

Restaurant: Jalsa
Location: Banashankari
Review data preview:
[('Rated 4.0', 'RATED\n  A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here again.'), ('Rated 4.0', 'RATED\n  I was here for dinner with my family on a weekday. The restaurant was completely empty. Ambience is good with some good old hindi music. Seating arrangement are good too. We ordered masala papad, panner and baby corn starters, lemon and corrionder soup, butter roti, olive and chilli paratha. Food was fresh and good, service is good too. Good for family hangout.\nCheers'), ('Rated 2.0', 'RATED\n  Its a restaurant near to Banashankari BDA. Me along with few of my office friends visited to have buffet but unfortunately they only provid

## Safely Parse Customer Reviews

In this step, we will parse the `reviews_list` column using Python's `ast.literal_eval()` to convert its string representation into Python lists safely.

We will extract individual review texts and their rating labels into separate columns. Empty or malformed entries will be handled without modifying the original dataset.

This creates a structured foundation for text cleaning, feature engineering, and NLP-based restaurant similarity.

In [37]:
import ast

def parse_reviews(review_string):
    try:
        reviews = ast.literal_eval(review_string)

        if not isinstance(reviews, list):
            return [], []

        review_texts = []
        review_ratings = []

        for review in reviews:
            if isinstance(review, (list, tuple)) and len(review) >= 2:
                rating_label = str(review[0]).strip()
                review_text = str(review[1]).strip()

                # Remove the repeated RATED prefix from review text
                review_text = review_text.removeprefix("RATED").strip()

                if review_text:
                    review_texts.append(review_text)
                    review_ratings.append(rating_label)

        return review_texts, review_ratings

    except (ValueError, SyntaxError, TypeError):
        return [], []


# Test the parser on one known restaurant entry
sample_review_string = df.loc[df["name"] == "Jalsa", "reviews_list"].iloc[0]

sample_texts, sample_ratings = parse_reviews(sample_review_string)

print("Number of extracted reviews:", len(sample_texts))
print("\nFirst extracted rating:", sample_ratings[0] if sample_ratings else "None")
print("\nFirst extracted review:")
print(sample_texts[0] if sample_texts else "No review extracted")

Number of extracted reviews: 12

First extracted rating: Rated 4.0

First extracted review:
A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here again.


## Validate the Review Parser

In this step, we will test the review parser on a sample of dataset rows, including entries with and without customer reviews.

We will check whether the parser returns valid lists, whether the extracted review and rating counts match, and whether any parsing errors occur. This validation helps ensure that the extraction logic is reliable before applying it to the entire dataset.

In [38]:
# Validate parsing on a sample of 100 rows
sample_df = df.sample(n=100, random_state=42)

validation_results = []

for index, row in sample_df.iterrows():
    review_texts, review_ratings = parse_reviews(row["reviews_list"])

    validation_results.append({
        "index": index,
        "name": row["name"],
        "input_length": len(row["reviews_list"]),
        "reviews_extracted": len(review_texts),
        "ratings_extracted": len(review_ratings),
        "counts_match": len(review_texts) == len(review_ratings)
    })

validation_df = pd.DataFrame(validation_results)

print("Rows tested:", len(validation_df))
print("Rows with extracted reviews:", (validation_df["reviews_extracted"] > 0).sum())
print("Rows with no extracted reviews:", (validation_df["reviews_extracted"] == 0).sum())
print("Rows where review and rating counts match:", validation_df["counts_match"].sum())

print("\nSample validation results:")
display(validation_df.head(10))

Rows tested: 100
Rows with extracted reviews: 81
Rows with no extracted reviews: 19
Rows where review and rating counts match: 100

Sample validation results:


,index,name,input_length,reviews_extracted,ratings_extracted,counts_match
0,8440,Kalingas,1505,5,5,True
1,23575,Angel Restaurant,675,2,2,True
2,16711,Cafe Talkhouse,797,2,2,True
3,42588,Paratha Xpress,1453,4,4,True
4,5266,Classic Lassi Shop,518,2,2,True
5,24663,CAD (M) CAD (B),678,3,3,True
6,21824,Cross Road Cafe,231,1,1,True
7,23715,Joey's,328,1,1,True
8,31303,Pipabu,2200,4,4,True
9,10675,Al Habibi Restaurant,774,7,7,True


## Extract Reviews Across the Full Dataset

Now that the parser has passed our initial validation, we will apply it to the complete dataset.

We will create two new columns: `review_texts`, containing the extracted customer review text, and `review_ratings`, containing the corresponding rating labels. The original `df` columns will remain unchanged.

After extraction, we will measure how many rows contain usable reviews and check whether the review and rating counts match across the entire dataset.

In [39]:
# Parse review data for every row
parsed_reviews = df["reviews_list"].apply(parse_reviews)

# Store extracted review text and rating labels in separate columns
df["review_texts"] = parsed_reviews.apply(lambda result: result[0])
df["review_ratings"] = parsed_reviews.apply(lambda result: result[1])

# Validate the full extraction
review_counts = df["review_texts"].str.len()
rating_counts = df["review_ratings"].str.len()

print("Total rows processed:", len(df))
print("Rows with extracted reviews:", (review_counts > 0).sum())
print("Rows with no extracted reviews:", (review_counts == 0).sum())
print("Total individual reviews extracted:", review_counts.sum())
print("Rows with matching review and rating counts:", (review_counts == rating_counts).sum())
print("Rows with mismatched counts:", (review_counts != rating_counts).sum())

Total rows processed: 51717
Rows with extracted reviews: 44117
Rows with no extracted reviews: 7600
Total individual reviews extracted: 1317680
Rows with matching review and rating counts: 51717
Rows with mismatched counts: 0


## Inspect Extracted Review Ratings

In this step, we will examine the rating labels extracted alongside the customer reviews.

We will count the different rating labels and identify any unexpected or missing values. This will help us determine whether the extracted ratings can be used reliably for review analysis and whether any cleaning is required before building the recommendation model.

In [40]:
# Combine all extracted rating labels into one Series
all_review_ratings = df["review_ratings"].explode().dropna()

print("Total extracted rating labels:", len(all_review_ratings))
print("\nRating label counts:")
display(all_review_ratings.value_counts().to_frame(name="Count"))

print("\nMissing or blank rating labels:")
print(
    all_review_ratings.isna().sum()
    + (all_review_ratings.astype(str).str.strip() == "").sum()
)

Total extracted rating labels: 1317680

Rating label counts:


,Count
review_ratings,
Rated 4.0,402035
Rated 5.0,379043
Rated 3.0,180661
Rated 1.0,153468
Rated 2.0,66995
Rated 3.5,56672
Rated 4.5,51197
Rated 2.5,17875
Rated 1.5,9651



Missing or blank rating labels:
0


## Check Duplicate Customer Reviews

In this step, we will inspect the extracted customer reviews for duplicate text.

Duplicate reviews can occur in the dataset and may unintentionally give certain opinions more influence when we calculate restaurant similarity. We will measure the extent of duplication before deciding how to handle it during text preparation.

This check will not remove or modify any data.

In [41]:
# Combine extracted review texts into one Series
all_review_texts = df["review_texts"].explode().dropna()

# Normalize whitespace and letter case for duplicate detection
normalized_reviews = (
    all_review_texts
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Total extracted review texts:", len(all_review_texts))
print("Unique review texts after basic normalization:", normalized_reviews.nunique())
print("Duplicate review occurrences:", normalized_reviews.duplicated().sum())
print(
    "Percentage of duplicate review occurrences:",
    round(normalized_reviews.duplicated().mean() * 100, 2), "%"
)

Total extracted review texts: 1317680
Unique review texts after basic normalization: 138097
Duplicate review occurrences: 1179583
Percentage of duplicate review occurrences: 89.52 %


## Investigate Duplicate Reviews Across Restaurants

In this step, we will identify frequently repeated customer review texts and examine how many different restaurant names and locations they are associated with.

This will help us understand whether duplication is concentrated within individual outlets or occurs across multiple restaurants. We will use the findings to choose an appropriate deduplication strategy without deleting legitimate review information prematurely.

In [42]:
# Create a review-level dataset for investigating duplicates
review_records = df[["name", "location", "review_texts"]].explode("review_texts")

review_records = review_records.dropna(subset=["review_texts"]).copy()

review_records["normalized_review"] = (
    review_records["review_texts"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Find the most frequently repeated normalized review texts
duplicate_summary = (
    review_records
    .groupby("normalized_review")
    .agg(
        occurrence_count=("normalized_review", "size"),
        restaurant_count=("name", "nunique"),
        location_count=("location", "nunique")
    )
    .sort_values("occurrence_count", ascending=False)
)

print("Top 10 most repeated review texts:")
display(duplicate_summary.head(10))

print("\nExamples of the most repeated review text:")
top_reviews = duplicate_summary.head(5).index

display(
    review_records.loc[
        review_records["normalized_review"].isin(top_reviews),
        ["name", "location", "review_texts", "normalized_review"]
    ].head(15)
)

Top 10 most repeated review texts:


,occurrence_count,restaurant_count,location_count
normalized_review,,,
good,18807,496,67
nice,2796,179,46
good food,2070,160,48
super,1307,107,41
good taste,1086,75,29
nice food,1052,73,33
not good,1015,98,41
very good,1009,101,38
awesome,847,76,31



Examples of the most repeated review text:


,name,location,review_texts,normalized_review
26,Coffee Tindi,Banashankari,Good,good
26,Coffee Tindi,Banashankari,good,good
26,Coffee Tindi,Banashankari,nice,nice
26,Coffee Tindi,Banashankari,nice,nice
26,Coffee Tindi,Banashankari,nice,nice
26,Coffee Tindi,Banashankari,nice,nice
26,Coffee Tindi,Banashankari,nice,nice
26,Coffee Tindi,Banashankari,good,good
26,Coffee Tindi,Banashankari,good,good
26,Coffee Tindi,Banashankari,good,good


## Measure Review Quality Before Cleaning

In this step, we will examine the distribution of customer review lengths after normalizing whitespace and letter case.

Very short reviews often contain generic feedback, such as “good” or “nice,” which provides limited information for identifying restaurants with similar customer experiences. We will measure how common these reviews are before setting a minimum length for the modeling dataset.

No reviews will be deleted in this step.

In [43]:
# Review-level dataset for quality analysis
review_quality = df[["name", "location", "review_texts"]].explode("review_texts")

review_quality = review_quality.dropna(subset=["review_texts"]).copy()

review_quality["normalized_review"] = (
    review_quality["review_texts"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

review_quality["review_length"] = review_quality["normalized_review"].str.len()

print("Review length statistics:")
display(review_quality["review_length"].describe().to_frame().T)

print("\nReviews of 1–5 characters:")
print((review_quality["review_length"].between(1, 5)).sum())

print("\nReviews of 1–10 characters:")
print((review_quality["review_length"].between(1, 10)).sum())

print("\nExamples of the shortest reviews:")
display(
    review_quality.nsmallest(10, "review_length")[
        ["name", "location", "review_texts", "review_length"]
    ]
)

Review length statistics:


,count,mean,std,min,25%,50%,75%,max
review_length,1317680.0,323.431471,371.209986,1.0,160.0,253.0,433.0,39304.0



Reviews of 1–5 characters:
30427

Reviews of 1–10 characters:
53002

Examples of the shortest reviews:


,name,location,review_texts,review_length
31,Foodiction,Banashankari,i,1
31,Foodiction,Banashankari,T,1
31,Foodiction,Banashankari,i,1
31,Foodiction,Banashankari,T,1
31,Foodiction,Banashankari,n,1
31,Foodiction,Banashankari,n,1
31,Foodiction,Banashankari,.,1
31,Foodiction,Banashankari,.,1
31,Foodiction,Banashankari,.,1
261,Karavali Lunch Home,Kumaraswamy Layout,I,1


## Investigate Unexpected Single-Character Reviews

In this step, we will investigate records where the extracted review text contains only one character.

These entries may result from an unexpected structure in the original review data. We will compare the original `reviews_list` value with the parsed output to determine whether the parser is handling every review format correctly.

This check will help us fix any parsing issues before applying text-cleaning rules.

In [44]:
# Find rows containing extracted reviews of one character
single_character_reviews = review_quality.loc[review_quality["review_length"] == 1].head(5)

for index, row in single_character_reviews.iterrows():
    print(f"DataFrame index: {index}")
    print(f"Restaurant: {row['name']}")
    print(f"Extracted review: {row['review_texts']!r}")

    print("\nOriginal review data:")
    print(df.loc[index, "reviews_list"][:1500])

    print("\nParsed reviews for this row:")
    print(df.loc[index, "review_texts"][:15])

    print("-" * 80)

DataFrame index: 31
Restaurant: Foodiction
Extracted review: 'i'

Original review data:
[('Rated 1.0', "RATED\n  Worst restaurant ever , Veg Manchurian worth ?130 doesn't even fill a tiny cup size. DO NOT order food fro. this place is a rip off with no regard for the money they charge.\nWorst restaurant even for zomato."), ('Rated 1.0', "RATED\n  I'm very sorry to say that you people lack in hygiene I can upload the photo of biryani rice which I ordered just now having a small cockroach in it.it was really disgusting and I'll surely not order anything or recommend anything from your restaurant but please maintain hygiene for the customer who comes to you .I won't say anything about taste as the portion what I ate was good but I became completely disappointed when I saw a dead baby cockroach."), ('Rated 1.0', 'RATED\n  The worst ever chicken biriyani I ordered, it was just plain chicken fried rice with one piece of chicken and believe me it was pathetic, kindly do not order again from h

## Verify the Parsed Review Column

In this step, we will verify the data type and structure of the `review_texts` column in the original DataFrame.

We will check whether each row contains a list of complete reviews or whether any rows contain a single string. This will help us identify why single-character entries appeared in the review-length analysis and ensure that subsequent preprocessing uses the correct data structure.

In [45]:
# Inspect the parsed review column for Foodiction
foodiction_row = df.loc[df["name"] == "Foodiction"].iloc[0]

print("Data type of review_texts:", type(foodiction_row["review_texts"]))
print("Number of parsed reviews:", len(foodiction_row["review_texts"]))
print("\nFirst five parsed reviews:")

for review in foodiction_row["review_texts"][:5]:
    print(repr(review))

print("\nData type of review_quality review_texts:")
print(type(review_quality.iloc[0]["review_texts"]))

Data type of review_texts: <class 'list'>
Number of parsed reviews: 575

First five parsed reviews:
"Worst restaurant ever , Veg Manchurian worth ?130 doesn't even fill a tiny cup size. DO NOT order food fro. this place is a rip off with no regard for the money they charge.\nWorst restaurant even for zomato."
"I'm very sorry to say that you people lack in hygiene I can upload the photo of biryani rice which I ordered just now having a small cockroach in it.it was really disgusting and I'll surely not order anything or recommend anything from your restaurant but please maintain hygiene for the customer who comes to you .I won't say anything about taste as the portion what I ate was good but I became completely disappointed when I saw a dead baby cockroach."
'The worst ever chicken biriyani I ordered, it was just plain chicken fried rice with one piece of chicken and believe me it was pathetic, kindly do not order again from here, be hungry for the night but never ever refer foodiction.'

## Rebuild the Review-Level Dataset Correctly

In this step, we will rebuild the review-level dataset directly from the original DataFrame.

Each extracted customer review should occupy one row, alongside its restaurant name and location. We will verify that the review text remains a complete string and then recalculate review lengths.

This ensures that our review-quality analysis is based on the correct data structure before we decide on any cleaning rules.

In [46]:
# Rebuild the review-level dataset from the original DataFrame
review_quality = df[["name", "location", "review_texts"]].explode("review_texts")

review_quality = review_quality.dropna(subset=["review_texts"]).copy()

# Normalize text for analysis
review_quality["normalized_review"] = (
    review_quality["review_texts"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

review_quality["review_length"] = review_quality["normalized_review"].str.len()

print("Total review-level rows:", len(review_quality))
print("Review text data type:", review_quality["review_texts"].map(type).value_counts().to_dict())

print("\nUpdated review length statistics:")
display(review_quality["review_length"].describe().to_frame().T)

print("\nShortest review examples:")
display(
    review_quality.nsmallest(10, "review_length")[
        ["name", "location", "review_texts", "review_length"]
    ]
)

Total review-level rows: 1317680
Review text data type: {<class 'str'>: 1317680}

Updated review length statistics:


,count,mean,std,min,25%,50%,75%,max
review_length,1317680.0,323.431471,371.209986,1.0,160.0,253.0,433.0,39304.0



Shortest review examples:


,name,location,review_texts,review_length
31,Foodiction,Banashankari,i,1
31,Foodiction,Banashankari,T,1
31,Foodiction,Banashankari,i,1
31,Foodiction,Banashankari,T,1
31,Foodiction,Banashankari,n,1
31,Foodiction,Banashankari,n,1
31,Foodiction,Banashankari,.,1
31,Foodiction,Banashankari,.,1
31,Foodiction,Banashankari,.,1
261,Karavali Lunch Home,Kumaraswamy Layout,I,1


## Compare Original and Exploded Review Text

In this step, we will compare the first complete review stored in the original DataFrame with the corresponding entry in the exploded review-level dataset.

We will check the number of reviews in the original list, the length of the first review, and the actual text stored in the first exploded row. This direct comparison will help us identify whether the problem occurs during list extraction or in the intermediate dataset.

In [47]:
# Get the original Foodiction row
foodiction_index = df.index[df["name"] == "Foodiction"][0]
original_reviews = df.loc[foodiction_index, "review_texts"]

# Get the exploded rows for the same restaurant record
exploded_reviews = review_quality.loc[review_quality.index == foodiction_index, "review_texts"]

print("Original list contains:", len(original_reviews), "reviews")
print("First original review length:", len(original_reviews[0]))
print("First original review preview:", repr(original_reviews[0][:150]))

print("\nFirst exploded review length:", len(exploded_reviews.iloc[0]))
print("First exploded review preview:", repr(exploded_reviews.iloc[0][:150]))

print("\nDoes the first exploded review match the first original review?")
print(exploded_reviews.iloc[0] == original_reviews[0])

Original list contains: 575 reviews
First original review length: 207
First original review preview: "Worst restaurant ever , Veg Manchurian worth ?130 doesn't even fill a tiny cup size. DO NOT order food fro. this place is a rip off with no regard for"

First exploded review length: 207
First exploded review preview: "Worst restaurant ever , Veg Manchurian worth ?130 doesn't even fill a tiny cup size. DO NOT order food fro. this place is a rip off with no regard for"

Does the first exploded review match the first original review?
True


## Verify review indexing and short-text records

We will check whether the original DataFrame index is unique and inspect the shortest reviews directly from the exploded dataset. This will help us determine whether the issue affects the data or only our earlier inspection.

In [48]:
print("Is original DataFrame index unique?", df.index.is_unique)

print("\nShortest reviews in exploded dataset:")
print(
    review_quality[
        ["name", "location", "review_texts", "review_length"]
    ]
    .sort_values("review_length")
    .head(10)
    .to_string(index=True)
)

Is original DataFrame index unique? True

Shortest reviews in exploded dataset:
                 name         location review_texts  review_length
27294  Sai Mango Tree              BTM            I              1
27294  Sai Mango Tree              BTM            I              1
14162   Andhra Amigos  Electronic City            Y              1
41144   Coorg Diaries    Kaggadasapura            o              1
41144   Coorg Diaries    Kaggadasapura            o              1
41144   Coorg Diaries    Kaggadasapura            o              1
14549      Desi Dhaba  Electronic City            h              1
14549      Desi Dhaba  Electronic City            h              1
50698    Biryani Park      Brookefield            .              1
28980  Sai Mango Tree              BTM            g              1


## Trace a short review back to the original data

We'll inspect the original record for Sai Mango Tree at index 27294 and compare its first few reviews with the corresponding exploded records. This should help us identify where the discrepancy originates.

In [49]:
# Inspect the original record
original_reviews = df.loc[27294, "review_texts"]

print("Original review count:", len(original_reviews))
print("\nFirst 5 original reviews:")

for review in original_reviews[:5]:
    print(repr(review[:150]))

# Inspect the first 5 exploded records for the same index
print("\nFirst 5 exploded reviews:")

print(
    review_quality.loc[
        review_quality.index == 27294,
        "review_texts"
    ].head(5).tolist()
)

Original review count: 71

First 5 original reviews:
'You get nice biryani in the afternoon, not service is not good. They started Andra meals recently, it is good option to try. Place is very small'
"This is one of the hidden jewels in Bangalore. If you are in the mood for Bangalore best spicy birayani then look no further. It's hidden just opposit"
'Pocket friendly!! Taste matches with best Biryanis in Bangalore. went with friends and had great time. Love to keep coming or ordering online.'
'Excellent biryani and the chicken starters are amazing..... great quantity at a very reasonable price..... Love to go this place again and again.... B'
'Having Mushroom Biryan here feel good both for your toung and tummy\n\nAwesome Mushroom biriyan in Bangalore cannot be compared with any of other restau'

First 5 exploded reviews:
['You get nice biryani in the afternoon, not service is not good. They started Andra meals recently, it is good option to try. Place is very small', "This is one of th

## Identify the source of the single-character reviews

We'll inspect the original row for one of the restaurants that appeared in the shortest-review results and check whether its exploded records contain the same short text. This will help us decide whether to filter very short reviews during preprocessing.

In [50]:
# Inspect the original record for Sai Mango Tree
short_review_index = 27294

original_reviews = df.loc[short_review_index, "review_texts"]

print("Single-character reviews in original list:")
for review in original_reviews:
    if len(review.strip()) <= 1:
        print(repr(review))

print("\nSingle-character reviews in exploded data:")
exploded_reviews = review_quality.loc[
    review_quality.index == short_review_index,
    "review_texts"
]

print(
    exploded_reviews[
        exploded_reviews.str.strip().str.len() <= 1
    ].tolist()
)

Single-character reviews in original list:
'I'
'g'
'I'
'g'
'I'
'g'
'I'
'g'

Single-character reviews in exploded data:
['I', 'g', 'I', 'g', 'I', 'g', 'I', 'g']


## Remove very short and duplicate reviews

We'll create a separate review-level dataset, leaving the original df unchanged. For now, we'll retain reviews with at least 10 characters after trimming whitespace and remove exact normalized duplicates within each restaurant outlet (name + location).

In [51]:
# Create a separate review-level dataset
review_data = df[["name", "location", "review_texts"]].explode("review_texts").dropna(subset=["review_texts"]).copy()

# Normalize review text for quality checks and deduplication
review_data["clean_review"] = (
    review_data["review_texts"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove empty and very short reviews
review_data = review_data[review_data["clean_review"].str.len() >= 10].copy()

# Remove duplicate reviews within the same outlet
review_data = review_data.drop_duplicates(subset=["name", "location", "clean_review"]).copy()

print("Review records after cleaning:", len(review_data))
print("Unique restaurant outlets:", review_data[["name", "location"]].drop_duplicates().shape[0])
print("Remaining missing review texts:", review_data["clean_review"].isna().sum())
print("Minimum review length:", review_data["clean_review"].str.len().min())
print("Remaining duplicate outlet-review pairs:",
      review_data.duplicated(subset=["name", "location", "clean_review"]).sum())

Review records after cleaning: 139832
Unique restaurant outlets: 9836
Remaining missing review texts: 0
Minimum review length: 10
Remaining duplicate outlet-review pairs: 0


## Analyze the number of reviews available per outlet

We need to understand whether most outlets have enough unique reviews to represent their customer experience. Outlets with very few reviews may produce less reliable recommendations.

In [52]:
reviews_per_outlet = (review_data.groupby(["name", "location"])["clean_review"].count())

print("Review count distribution per outlet:")
print(reviews_per_outlet.describe().round(2))

print("\nOutlets with fewer than 3 reviews:",
      (reviews_per_outlet < 3).sum())

print("Outlets with at least 5 reviews:",
      (reviews_per_outlet >= 5).sum())

print("Outlets with at least 10 reviews:",
      (reviews_per_outlet >= 10).sum())

Review count distribution per outlet:
count    9836.00
mean       14.22
std        27.80
min         1.00
25%         2.00
50%         5.00
75%        14.00
max       481.00
Name: clean_review, dtype: float64

Outlets with fewer than 3 reviews: 3059
Outlets with at least 5 reviews: 5128
Outlets with at least 10 reviews: 3250


## Filter outlets with sufficient review data

We'll retain outlets with at least five cleaned, unique reviews and aggregate their reviews into one text document per outlet. This outlet-level dataset will be the input for our TF-IDF similarity model.

In [53]:
# Identify outlets with at least 5 cleaned reviews
eligible_outlets = reviews_per_outlet[reviews_per_outlet >= 5].index

# Keep reviews belonging to eligible outlets
model_reviews = review_data.set_index(["name", "location"]).loc[eligible_outlets].reset_index()

# Combine reviews into one document per outlet
outlet_reviews = (
    model_reviews.groupby(["name", "location"])["clean_review"]
    .agg(" ".join)
    .reset_index(name="combined_reviews")
)

print("Eligible outlets:", len(outlet_reviews))
print("Unique outlet identities:",
      outlet_reviews[["name", "location"]].drop_duplicates().shape[0])
print("Missing combined review documents:",
      outlet_reviews["combined_reviews"].isna().sum())

print("\nCombined review length summary:")
print(outlet_reviews["combined_reviews"].str.len().describe().round(0))

Eligible outlets: 5128
Unique outlet identities: 5128
Missing combined review documents: 0

Combined review length summary:
count      5128.0
mean       8418.0
std       11801.0
min         140.0
25%        2304.0
50%        4302.0
75%        9246.0
max      134637.0
Name: combined_reviews, dtype: float64


## Inspect the restaurant rating values

The rate column contains values such as 3.9/5, NEW, and missing values. First, we'll inspect the current formats so we can convert ratings safely without accidentally treating unrated restaurants as having a zero rating.

In [54]:
print("Sample rating values:")
print(df["rate"].value_counts(dropna=False).head(20))

print("\nRating data type:", df["rate"].dtype)
print("Missing ratings:", df["rate"].isna().sum())

Sample rating values:
rate
NaN       7775
NEW       2208
3.9/5     2098
3.8/5     2022
3.7/5     2011
3.9 /5    1874
3.8 /5    1851
3.7 /5    1810
3.6/5     1773
4.0/5     1609
4.0 /5    1574
3.6 /5    1543
4.1/5     1474
4.1 /5    1474
3.5/5     1431
3.5 /5    1353
3.4/5     1259
3.4 /5    1217
3.3/5     1168
4.2 /5    1165
Name: count, dtype: int64

Rating data type: str
Missing ratings: 7775


## Convert ratings to numeric values

We'll create a separate ratings dataset, preserving the original df. First, we'll standardize the rating text and convert valid numeric values. Then we'll inspect the conversion results.

In [55]:
# Create a separate dataset for restaurant ratings
rating_data = df[["name", "location", "rate", "votes"]].copy()

# Standardize rating text and extract numeric values
rating_data["rating"] = pd.to_numeric(
    rating_data["rate"]
    .astype("string")
    .str.strip()
    .str.extract(r"^(\d+(?:\.\d+)?)\s*/\s*5$", expand=False),
    errors="coerce"
)

print("Total rating records:", len(rating_data))
print("Valid numeric ratings:", rating_data["rating"].notna().sum())
print("Missing or unrated records:", rating_data["rating"].isna().sum())

print("\nConverted rating examples:")
print(
    rating_data[["rate", "rating"]]
    .drop_duplicates()
    .head(15)
    .to_string(index=False)
)

Total rating records: 51717
Valid numeric ratings: 41665
Missing or unrated records: 10052

Converted rating examples:
 rate  rating
4.1/5     4.1
3.8/5     3.8
3.7/5     3.7
3.6/5     3.6
4.6/5     4.6
4.0/5     4.0
4.2/5     4.2
3.9/5     3.9
3.1/5     3.1
3.0/5     3.0
3.2/5     3.2
3.3/5     3.3
2.8/5     2.8
4.4/5     4.4
4.3/5     4.3


## Aggregate ratings and votes by restaurant outlet

We'll group by name and location, calculate the average valid rating, and sum the votes after converting them to numeric values. The vote total will be used as a supporting popularity measure, but we should verify that votes aren't duplicated across repeated rows.

In [56]:
# Convert vote counts to numeric values
rating_data["votes"] = pd.to_numeric(
    rating_data["votes"],
    errors="coerce"
)

# Aggregate ratings and votes by outlet
outlet_ratings = (
    rating_data.groupby(["name", "location"], dropna=False)
    .agg(
        average_rating=("rating", "mean"),
        rating_records=("rating", "count"),
        average_votes=("votes", "mean"),
        max_votes=("votes", "max")
    )
    .reset_index()
)

print("Unique outlet rating records:", len(outlet_ratings))
print("Outlets with a valid average rating:",
      outlet_ratings["average_rating"].notna().sum())

print("\nRating summary:")
print(outlet_ratings["average_rating"].describe().round(2))

print("\nSample aggregated ratings:")
print(
    outlet_ratings.sort_values(
        "average_rating", ascending=False, na_position="last"
    ).head(10).to_string(index=False)
)

Unique outlet rating records: 12151
Outlets with a valid average rating: 9249

Rating summary:
count    9249.0
mean       3.63
std        0.43
min         1.8
25%         3.3
50%        3.67
75%         3.9
max         4.9
Name: average_rating, dtype: Float64

Sample aggregated ratings:
                                                                            name              location  average_rating  rating_records  average_votes  max_votes
                                                                Milano Ice Cream           Indiranagar             4.9               1    2090.000000       2090
SantÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ© Spa Cuisine           Indiranagar             4.9               1     246.000000        246
                                                       AB's - Absolute Barbecues                   BTM             4.9               7    6414.857143       6490
                                                          Belgian Wa

## Create the final outlet-level modeling dataset

We'll merge outlet_reviews with outlet_ratings using restaurant name and location. We'll retain outlets with sufficient review text even if their rating is missing. Those outlets can still be considered for text similarity, although they cannot be ranked by a known rating.

In [57]:
# Merge cleaned review documents with outlet ratings
restaurant_data = outlet_reviews.merge(
    outlet_ratings,
    on=["name", "location"],
    how="left",
    validate="one_to_one"
)

print("Total outlets in modeling dataset:", len(restaurant_data))
print("Missing ratings:", restaurant_data["average_rating"].isna().sum())
print("Missing vote counts:", restaurant_data["max_votes"].isna().sum())
print("Duplicate outlet identities:",
      restaurant_data.duplicated(subset=["name", "location"]).sum())

print("\nModeling dataset preview:")
print(
    restaurant_data[
        ["name", "location", "average_rating", "max_votes", "combined_reviews"]
    ]
    .head(5)
    .to_string(index=False)
)

Total outlets in modeling dataset: 5128
Missing ratings: 53
Missing vote counts: 0
Duplicate outlet identities: 0

Modeling dataset preview:
                                name              location  average_rating  max_votes                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

## Convert restaurant reviews into TF-IDF vectors

We'll use TfidfVectorizer with unigrams and bigrams. This captures individual words as well as two-word phrases, such as good food or friendly staff. We'll also remove common English stop words and very rare or overly common terms.

In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

# Convert combined restaurant reviews into TF-IDF vectors
tfidf_matrix = tfidf_vectorizer.fit_transform(
    restaurant_data["combined_reviews"]
)

print("Number of restaurant outlets:", tfidf_matrix.shape[0])
print("Number of TF-IDF features:", tfidf_matrix.shape[1])
print("Non-zero values in matrix:", tfidf_matrix.nnz)
print("Matrix density (%):",
      round(tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]) * 100, 4))

Number of restaurant outlets: 5128
Number of TF-IDF features: 50000
Non-zero values in matrix: 2615302
Matrix density (%): 1.02


## Calculate cosine similarity between restaurants

Cosine similarity measures how similar two restaurant review vectors are. Scores closer to 1 indicate greater similarity, while scores closer to 0 indicate less similarity.

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

# Select the first restaurant as a test case
test_index = 0

# Calculate similarity between the selected restaurant and all outlets
similarity_scores = cosine_similarity(
    tfidf_matrix[test_index],
    tfidf_matrix
).flatten()

# Get the 10 most similar outlets, excluding the selected restaurant
similar_indices = similarity_scores.argsort()[::-1][1:11]

test_restaurant = restaurant_data.iloc[test_index]["name"]

print("Test restaurant:", test_restaurant)
print("\nTop 10 similar outlets:")

print(
    restaurant_data.iloc[similar_indices][
        ["name", "location", "average_rating", "max_votes"]
    ].assign(
        similarity=similarity_scores[similar_indices]
    ).to_string(index=False)
)

Test restaurant: #FeelTheROLL

Top 10 similar outlets:
                      name          location  average_rating  max_votes  similarity
Kolkata Famous Katti Rolls      Marathahalli             3.4          8    0.170272
Kolkata Famous Katti Rolls      New BEL Road             3.5         22    0.150674
           Rolls On Wheels               BTM        3.266667        127    0.139467
      Kolkata Kathi Roll's      Banashankari             3.6         18    0.132053
Kolkata Famous Kathi Rolls       Brookefield             3.6         37    0.131234
               Kusum Rolls      New BEL Road            3.55         76    0.128244
 Kolkata Famous Kati Rolls Bannerghatta Road             3.1         25    0.117911
              Le Roll Cafe      Kammanahalli             3.9         57    0.111725
Kolkata Famous Kathi Rolls   Electronic City             3.5         14    0.108559
               Hungry Lion     Wilson Garden             3.0         16    0.108466


## Create the restaurant recommendation function

The function will:
1. Accept a restaurant name as input.
2. Find its review-based similarity scores against other outlets.
3. Exclude the selected outlet itself.
4. Rank the most similar candidates by their average rating, using similarity as a secondary ranking criterion.
5. Return the restaurant name, location, rating, votes, and similarity score.
We'll initially retrieve the 50 most similar candidates before ranking them by rating. This prevents the final list from being limited to only the highest similarity scores.

In [60]:
def recommend_restaurants(restaurant_name, top_n=10, candidate_count=50):
    # Find matching restaurant names
    matching_indices = restaurant_data.index[
        restaurant_data["name"].str.casefold()
        == restaurant_name.strip().casefold()
    ].tolist()

    if not matching_indices:
        print("Restaurant not found. Please check the name.")
        return

    # Use the first matching outlet if a name exists at multiple locations
    selected_index = matching_indices[0]

    # Calculate similarity against all outlets
    similarity_scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix
    ).flatten()

    # Exclude the selected outlet itself
    similarity_scores[selected_index] = -1

    # Select the most similar candidate outlets
    similar_indices = similarity_scores.argsort()[::-1][:candidate_count]

    recommendations = restaurant_data.iloc[similar_indices].copy()
    recommendations["similarity"] = similarity_scores[similar_indices]

    # Exclude candidates without a valid rating for rating-based ranking
    recommendations = recommendations.dropna(subset=["average_rating"])

    # Rank primarily by rating, then by similarity
    recommendations = recommendations.sort_values(
        ["average_rating", "similarity"],
        ascending=[False, False]
    )

    return recommendations[
        ["name", "location", "average_rating", "max_votes", "similarity"]
    ].head(top_n).reset_index(drop=True)

## Test the recommendation function

We'll request the top 10 recommendations for #FeelTheROLL. The function will first identify similar restaurants from their reviews, then rank the candidate restaurants by average rating, using similarity to break ties.

In [61]:
recommendations = recommend_restaurants("#FeelTheROLL", top_n=10)

print(recommendations.to_string(index=False))

                    name      location  average_rating  max_votes  similarity
    Siddiqu Kabab Center Richmond Road            4.44        181    0.080841
          Al Amanah Cafe  Kammanahalli             4.4       1514    0.092224
          Kitchen of Joy   Indiranagar            4.15        581    0.076332
Charminar Kabab Paradise   Frazer Town             4.0        106    0.087488
     K.G.N Kabbab Corner  Kalyan Nagar             4.0         91    0.076623
                Eggzotic        Domlur            3.95        391    0.078052
            Le Roll Cafe  Kammanahalli             3.9         57    0.111725
             Pathaan Sir           BTM             3.9        184    0.091748
                  Fanoos   Frazer Town             3.9        132    0.091559
Savji's Roll 'N' Biryani   Vijay Nagar             3.9        441    0.086636


## Compare similarity-first and rating-first recommendations

We'll retrieve the 10 most similar restaurants and display their ratings alongside their similarity scores. This will help us decide how to balance relevance with customer ratings.

In [62]:
# Find the selected restaurant
selected_index = restaurant_data.index[
    restaurant_data["name"].str.casefold() == "#feeltheroll".casefold()][0]

# Calculate similarity against all outlets
similarity_scores = cosine_similarity(
    tfidf_matrix[selected_index],
    tfidf_matrix
).flatten()

similarity_scores[selected_index] = -1

# Get the 10 most similar outlets
similar_indices = similarity_scores.argsort()[::-1][:10]

similarity_first = restaurant_data.iloc[similar_indices][
    ["name", "location", "average_rating", "max_votes"]
].copy()

similarity_first["similarity"] = similarity_scores[similar_indices]

print(
    similarity_first.sort_values(
        "similarity", ascending=False
    ).to_string(index=False)
)

                      name          location  average_rating  max_votes  similarity
Kolkata Famous Katti Rolls      Marathahalli             3.4          8    0.170272
Kolkata Famous Katti Rolls      New BEL Road             3.5         22    0.150674
           Rolls On Wheels               BTM        3.266667        127    0.139467
      Kolkata Kathi Roll's      Banashankari             3.6         18    0.132053
Kolkata Famous Kathi Rolls       Brookefield             3.6         37    0.131234
               Kusum Rolls      New BEL Road            3.55         76    0.128244
 Kolkata Famous Kati Rolls Bannerghatta Road             3.1         25    0.117911
              Le Roll Cafe      Kammanahalli             3.9         57    0.111725
Kolkata Famous Kathi Rolls   Electronic City             3.5         14    0.108559
               Hungry Lion     Wilson Garden             3.0         16    0.108466


## Update the recommendation function

We will follow a two-stage recommendation approach:
1. Similarity filtering: Select the 20 restaurants with the most similar customer reviews.
2. Rating-based ranking: Sort these restaurants by average rating, using similarity as the secondary sorting criterion.
This should improve relevance while still prioritizing highly rated restaurants.

In [63]:
def recommend_restaurants(restaurant_name, top_n=10, candidate_count=20):
    # Find matching restaurant names
    matching_indices = restaurant_data.index[
        restaurant_data["name"].str.casefold()
        == restaurant_name.strip().casefold()
    ].tolist()

    if not matching_indices:
        print("Restaurant not found. Please check the name.")
        return

    # Select the first matching outlet
    selected_index = matching_indices[0]

    # Calculate similarity against all outlets
    similarity_scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix
    ).flatten()

    # Exclude the selected outlet itself
    similarity_scores[selected_index] = -1

    # Select the most similar candidate outlets
    similar_indices = similarity_scores.argsort()[::-1][:candidate_count]

    recommendations = restaurant_data.iloc[similar_indices].copy()
    recommendations["similarity"] = similarity_scores[similar_indices]

    # Keep restaurants with valid ratings
    recommendations = recommendations.dropna(subset=["average_rating"])

    # Rank by rating first, then similarity
    recommendations = recommendations.sort_values(
        ["average_rating", "similarity"],
        ascending=[False, False]
    )

    return recommendations[
        ["name", "location", "average_rating", "max_votes", "similarity"]
    ].head(top_n).reset_index(drop=True)


# Test the updated function
recommend_restaurants("#FeelTheROLL", top_n=10)

,name,location,average_rating,max_votes,similarity
0,Le Roll Cafe,Kammanahalli,3.9,57,0.111725
1,Sahana's (Nati Style),Wilson Garden,3.725,50,0.106365
2,Tayabbs,Ulsoor,3.7,71,0.104691
3,Nizaam's Kaati Rolls & Kebabs,Jeevan Bhima Nagar,3.7,112,0.098164
4,Kolkata Famous Kati Rolls,Bellandur,3.65,19,0.108205
5,Kolkata Kathi Roll's,Banashankari,3.6,18,0.132053
6,Kolkata Famous Kathi Rolls,Brookefield,3.6,37,0.131234
7,Rolls Corner,Basavanagudi,3.6,22,0.106293
8,What the Food,BTM,3.6,38,0.098760
9,Kusum Rolls,New BEL Road,3.55,76,0.128244


## Test the model with another restaurant

We will test the recommendation function using another restaurant, Kusum Rolls, to see whether the model identifies relevant alternatives beyond our first example.

In [64]:
recommend_restaurants("Kusum Rolls", top_n=10)

,name,location,average_rating,max_votes,similarity
0,Magix's Parattha Roll,BTM,3.75,186,0.206721
1,Tayabbs,Ulsoor,3.7,71,0.175495
2,Nizaam's Kaati Rolls & Kebabs,Jeevan Bhima Nagar,3.7,112,0.208211
3,Rolls Plaza,Marathahalli,3.7,63,0.177376
4,Kolkata Famous Kati Rolls,Bellandur,3.65,19,0.181543
5,Kolkata Famous Kathi Rolls,Brookefield,3.6,37,0.246708
6,Kolkata Kathi Roll's,Banashankari,3.6,18,0.198414
7,Rolls Corner,Basavanagudi,3.6,22,0.171220
8,Kolkata Famous Katti Rolls,New BEL Road,3.5,22,0.247501
9,Kolkata Famous Kathi Rolls,Electronic City,3.5,14,0.181998


## Check restaurant names with multiple outlets

We will identify restaurant names associated with multiple locations. This will help us decide how the recommendation system should handle restaurant names that are not unique.

In [65]:
# Count the number of outlets for each restaurant name
outlet_counts = (
    restaurant_data.groupby("name")["location"]
    .nunique()
    .sort_values(ascending=False)
)

print("Restaurant names with multiple outlets:", (outlet_counts > 1).sum())
print("\nTop 10 restaurant names by outlet count:")
print(outlet_counts.head(10).to_string())

Restaurant names with multiple outlets: 522

Top 10 restaurant names by outlet count:
name
Faasos                      23
Five Star Chicken           19
Cafe Coffee Day             18
KFC                         17
Sweet Truth                 16
Corner House Ice Cream      15
Empire Restaurant           15
Mojo Pizza - 2X Toppings    14
Beijing Bites               14
Polar Bear                  14


## Add location support to the recommendation function

We will improve the function to:
- Accept an optional restaurant location.
- Display available locations when a restaurant name matches multiple outlets.
- Calculate recommendations only after identifying the correct outlet.
- Continue ranking the 20 most similar candidates by rating.

In [66]:
def recommend_restaurants(restaurant_name, location=None, top_n=10, candidate_count=20):
    # Find matching restaurant names
    matches = restaurant_data[
        restaurant_data["name"].str.casefold()
        == restaurant_name.strip().casefold()
    ]

    if matches.empty:
        print("Restaurant not found. Please check the name.")
        return

    # Handle restaurants with multiple outlets
    if location is None and len(matches) > 1:
        print("Multiple outlets found. Please specify a location:")
        print(matches["location"].to_string(index=False))
        return

    if location is not None:
        matches = matches[
            matches["location"].str.casefold()
            == location.strip().casefold()
        ]

        if matches.empty:
            print("Restaurant not found at the specified location.")
            return

    selected_index = matches.index[0]

    # Calculate similarity
    similarity_scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix
    ).flatten()

    similarity_scores[selected_index] = -1

    # Select the most similar candidates
    similar_indices = similarity_scores.argsort()[::-1][:candidate_count]

    recommendations = restaurant_data.iloc[similar_indices].copy()
    recommendations["similarity"] = similarity_scores[similar_indices]

    # Keep restaurants with valid ratings
    recommendations = recommendations.dropna(subset=["average_rating"])

    # Rank by rating and similarity
    recommendations = recommendations.sort_values(
        ["average_rating", "similarity"],
        ascending=[False, False]
    )

    return recommendations[
        ["name", "location", "average_rating", "max_votes", "similarity"]
    ].head(top_n).reset_index(drop=True)


# Test with a restaurant that has multiple outlets
recommend_restaurants("Faasos")

Multiple outlets found. Please specify a location:
                       BTM
              Banashankari
         Bannerghatta Road
                 Bellandur
           Electronic City
               Frazer Town
                       HSR
                Hosur Road
ITPL Main Road, Whitefield
               Indiranagar
                  JP Nagar
              Kammanahalli
     Koramangala 1st Block
     Koramangala 8th Block
              Marathahalli
          Old Airport Road
               Rajajinagar
            Residency Road
              Sanjay Nagar
             Sarjapur Road
              Thippasandra
             Vasanth Nagar
                Whitefield


## Test recommendations for a specific outlet

We will now select Faasos at Indiranagar and generate recommendations using that outlet's customer reviews.
The function should identify the exact outlet, calculate review similarity, select the 20 most similar restaurants, and return the 10 highest-rated candidates.

In [67]:
recommend_restaurants(
    restaurant_name="Faasos",
    location="Indiranagar",
    top_n=10
)

,name,location,average_rating,max_votes,similarity
0,Faasos,Banashankari,4.2,415,0.290880
1,Faasos,Thippasandra,4.1,112,0.315318
2,Faasos,Frazer Town,4.1,38,0.215539
3,Faasos,"ITPL Main Road, Whitefield",4.0,176,0.345005
4,Faasos,Hosur Road,4.0,138,0.294148
5,Faasos,Bellandur,4.0,193,0.285569
6,Faasos,Koramangala 8th Block,4.0,463,0.280349
7,Faasos,Koramangala 1st Block,4.0,67,0.266463
8,Faasos,Whitefield,4.0,206,0.264825
9,Faasos,Electronic City,4.0,506,0.226842


## Exclude the same restaurant from recommendations

We will update the recommendation function to exclude all outlets with the same restaurant name as the selected restaurant.
The model will then:
1. Identify the correct restaurant and location.
2. Calculate review similarity.
3. Exclude all outlets of the selected restaurant.
4. Select the 20 most similar alternatives.
5. Rank them by average rating, using similarity as the secondary criterion.

In [68]:
def recommend_restaurants(restaurant_name, location=None, top_n=10, candidate_count=20):
    # Find matching restaurant names
    matches = restaurant_data[
        restaurant_data["name"].str.casefold()
        == restaurant_name.strip().casefold()
    ]

    if matches.empty:
        print("Restaurant not found. Please check the name.")
        return

    # Handle restaurants with multiple outlets
    if location is None and len(matches) > 1:
        print("Multiple outlets found. Please specify a location:")
        print(matches["location"].to_string(index=False))
        return

    if location is not None:
        matches = matches[
            matches["location"].str.casefold()
            == location.strip().casefold()
        ]

        if matches.empty:
            print("Restaurant not found at the specified location.")
            return

    selected_index = matches.index[0]

    # Calculate similarity
    similarity_scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix
    ).flatten()

    # Exclude every outlet with the same restaurant name
    same_restaurant = (
        restaurant_data["name"].str.casefold()
        == restaurant_data.loc[selected_index, "name"].casefold()
    )

    similarity_scores[same_restaurant.to_numpy()] = -1

    # Select the most similar alternatives
    similar_indices = similarity_scores.argsort()[::-1][:candidate_count]

    recommendations = restaurant_data.iloc[similar_indices].copy()
    recommendations["similarity"] = similarity_scores[similar_indices]

    # Keep restaurants with valid ratings
    recommendations = recommendations.dropna(subset=["average_rating"])

    # Rank by rating first, then similarity
    recommendations = recommendations.sort_values(
        ["average_rating", "similarity"],
        ascending=[False, False]
    )

    return recommendations[
        ["name", "location", "average_rating", "max_votes", "similarity"]
    ].head(top_n).reset_index(drop=True)


# Test with Faasos at Indiranagar
recommend_restaurants("Faasos", location="Indiranagar", top_n=10)

,name,location,average_rating,max_votes,similarity
0,Al Amanah Cafe,Kammanahalli,4.4,1514,0.097320
1,eat.fit,Indiranagar,4.34,882,0.088317
2,BOX8- Desi Meals,Rajajinagar,4.1,559,0.111279
3,Leon Grill,Marathahalli,4.066667,671,0.096500
4,Food Springs,Jayanagar,4.0,233,0.086270
5,Leon Grill,New BEL Road,4.0,336,0.085269
6,Mc Donald's,Whitefield,3.9,116,0.097354
7,Saahib,HSR,3.9,89,0.093257
8,Desi Dhaba,Electronic City,3.8,303,0.084684
9,RoadHouse Grill,Whitefield,3.8,99,0.085985


## Inspect the most similar alternatives before rating-based ranking

We will examine the 10 most similar restaurants to Faasos (Indiranagar), excluding other Faasos outlets.
This will help us distinguish between two possibilities: the TF-IDF model is finding weak matches, or our rating-based ranking is pushing stronger matches further down the list.

In [69]:
# Identify the selected Faasos outlet
selected_index = restaurant_data.index[
    (restaurant_data["name"] == "Faasos")
    & (restaurant_data["location"] == "Indiranagar")
][0]

# Calculate similarity
similarity_scores = cosine_similarity(
    tfidf_matrix[selected_index],
    tfidf_matrix
).flatten()

# Exclude all Faasos outlets
same_restaurant = (
    restaurant_data["name"].str.casefold() == "faasos"
)
similarity_scores[same_restaurant.to_numpy()] = -1

# Retrieve the 10 most similar alternatives
similar_indices = similarity_scores.argsort()[::-1][:10]

similarity_first = restaurant_data.iloc[similar_indices][
    ["name", "location", "average_rating", "max_votes"]
].copy()

similarity_first["similarity"] = similarity_scores[similar_indices]

print(similarity_first.to_string(index=False))

                         name           location  average_rating  max_votes  similarity
             BOX8- Desi Meals        Rajajinagar             4.1        559    0.111279
                      Tayabbs             Ulsoor             3.7         71    0.102661
Nizaam's Kaati Rolls & Kebabs Jeevan Bhima Nagar             3.7        112    0.097833
                  Mc Donald's         Whitefield             3.9        116    0.097354
               Al Amanah Cafe       Kammanahalli             4.4       1514    0.097320
                Roll Junction         Whitefield             3.1          8    0.096871
                    Doner Bel       New BEL Road             3.4         64    0.096736
                   Leon Grill       Marathahalli        4.066667        671    0.096500
                   Kaati Zone      Church Street        3.063636        165    0.096213
                       Saahib                HSR             3.9         89    0.093257


## Test recommendations across different restaurant categories

We will test the recommendation system using three different restaurants: KFC, Cafe Coffee Day, and Corner House Ice Cream.
For each restaurant, we'll select an available outlet and examine whether the recommendations reflect its food category. This is a qualitative evaluation of our content-based recommendation model.

In [70]:
# Select one available outlet for each restaurant
test_restaurants = [
    "KFC",
    "Cafe Coffee Day",
    "Corner House Ice Cream"
]

for restaurant_name in test_restaurants:
    outlet = restaurant_data[
        restaurant_data["name"] == restaurant_name
    ].iloc[0]

    print(f"\n{'=' * 60}")
    print(f"Restaurant: {outlet['name']}")
    print(f"Location: {outlet['location']}")
    print(f"{'=' * 60}")

    recommendations = recommend_restaurants(
        restaurant_name=outlet["name"],
        location=outlet["location"],
        top_n=5
    )

    print(recommendations.to_string(index=False))


Restaurant: KFC
Location: Basavanagudi
            name              location  average_rating  max_votes  similarity
        Truffles Koramangala 5th Block             4.7      14726    0.153694
BOX8- Desi Meals           Rajajinagar             4.1        559    0.159522
      Leon Grill          Marathahalli        4.066667        671    0.154594
      Whooppeezz                   BTM             4.0        374    0.153952
     Big Chicken          Kammanahalli             3.9        118    0.170282

Restaurant: Cafe Coffee Day
Location: Banashankari
                   name    location  average_rating  max_votes  similarity
               Truffles Indiranagar             4.6       1313    0.135258
              Zee5 Loft         HSR             4.4        798    0.137696
Alchemy Coffee Roasters   Jayanagar             4.2        737    0.141130
          Matteo Coffea Indiranagar             4.2        858    0.137978
       Glen's Bakehouse  Whitefield             4.2        816   

## Check duplicate restaurant brands in recommendations

Before modifying our recommendation function, we will measure how often the current recommendations contain repeated restaurant names.
We will test KFC, Cafe Coffee Day, and Corner House Ice Cream using 10 recommendations each.

In [71]:
# Evaluate repeated restaurant names in recommendations
test_restaurants = [
    ("KFC", "Basavanagudi"),
    ("Cafe Coffee Day", "Banashankari"),
    ("Corner House Ice Cream", "Banashankari")
]

for restaurant_name, location in test_restaurants:
    recommendations = recommend_restaurants(
        restaurant_name=restaurant_name,
        location=location,
        top_n=10
    )

    total_recommendations = len(recommendations)
    unique_restaurants = recommendations["name"].nunique()
    repeated_outlets = total_recommendations - unique_restaurants

    print(f"\nRestaurant: {restaurant_name}")
    print(f"Total recommendations: {total_recommendations}")
    print(f"Unique restaurant names: {unique_restaurants}")
    print(f"Repeated outlets: {repeated_outlets}")


Restaurant: KFC
Total recommendations: 10
Unique restaurant names: 10
Repeated outlets: 0

Restaurant: Cafe Coffee Day
Total recommendations: 10
Unique restaurant names: 10
Repeated outlets: 0

Restaurant: Corner House Ice Cream
Total recommendations: 10
Unique restaurant names: 3
Repeated outlets: 7


## Add unique restaurant recommendations

We will update the recommendation function to return unique restaurant names.
The model will first identify similar restaurants, rank them by customer rating, and then retain only the highest-ranked outlet for each restaurant name.
We will also increase the candidate pool to 100 outlets so that removing duplicate brands does not unnecessarily reduce the number of recommendations.

In [72]:
def recommend_restaurants(restaurant_name, location=None, top_n=10, candidate_count=100):
    # Find matching restaurant names
    matches = restaurant_data[
        restaurant_data["name"].str.casefold()
        == restaurant_name.strip().casefold()
    ]

    if matches.empty:
        print("Restaurant not found. Please check the name.")
        return

    # Handle multiple outlets
    if location is None and len(matches) > 1:
        print("Multiple outlets found. Please specify a location:")
        print(matches["location"].to_string(index=False))
        return

    if location is not None:
        matches = matches[
            matches["location"].str.casefold()
            == location.strip().casefold()
        ]

        if matches.empty:
            print("Restaurant not found at the specified location.")
            return

    selected_index = matches.index[0]

    # Calculate similarity
    similarity_scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix
    ).flatten()

    # Exclude all outlets of the selected restaurant
    same_restaurant = (
        restaurant_data["name"].str.casefold()
        == restaurant_data.loc[selected_index, "name"].casefold()
    )
    similarity_scores[same_restaurant.to_numpy()] = -1

    # Select similar candidates
    similar_indices = similarity_scores.argsort()[::-1][:candidate_count]

    recommendations = restaurant_data.iloc[similar_indices].copy()
    recommendations["similarity"] = similarity_scores[similar_indices]

    # Keep valid ratings
    recommendations = recommendations.dropna(subset=["average_rating"])

    # Rank by rating, then similarity
    recommendations = recommendations.sort_values(
        ["average_rating", "similarity"],
        ascending=[False, False]
    )

    # Keep only the highest-ranked outlet per restaurant
    recommendations["normalized_name"] = (
        recommendations["name"].str.casefold().str.strip()
    )

    recommendations = recommendations.drop_duplicates(
        subset=["normalized_name"],
        keep="first"
    )

    return recommendations[
        ["name", "location", "average_rating", "max_votes", "similarity"]
    ].head(top_n).reset_index(drop=True)


# Test recommendation diversity
recommend_restaurants(
    "Corner House Ice Cream",
    location="Banashankari",
    top_n=10
)

,name,location,average_rating,max_votes,similarity
0,Milano Ice Cream,Indiranagar,4.9,2090,0.112688
1,Truffles,Koramangala 5th Block,4.7,14726,0.094130
2,Mugful Of Stories,Kalyan Nagar,4.6,274,0.102524
3,Smoor,Indiranagar,4.6,1732,0.097899
4,Polar Bear,"Varthur Main Road, Whitefield",4.5,53,0.131976
5,Stoner,Koramangala 5th Block,4.495238,2668,0.090968
6,Black Deer,Koramangala 5th Block,4.4,149,0.112455
7,Art of Delight,Residency Road,4.4,1916,0.104865
8,Lavonne,Indiranagar,4.4,1132,0.090253
9,Mama Mia!,Indiranagar,4.3,429,0.111620


## Test a balanced recommendation score

We will experiment with a combined ranking score that considers both review similarity and customer ratings.
For this initial experiment:
- 70% weight: Review similarity
- 30% weight: Restaurant rating
We'll normalize similarity scores within the candidate pool and convert restaurant ratings to a 0–1 scale. These weights are experimental, not yet optimized.

In [73]:
# Test balanced ranking for Corner House Ice Cream
selected_index = restaurant_data.index[
    (restaurant_data["name"] == "Corner House Ice Cream")
    & (restaurant_data["location"] == "Banashankari")
][0]

similarity_scores = cosine_similarity(
    tfidf_matrix[selected_index],
    tfidf_matrix
).flatten()

# Exclude the selected restaurant brand
same_restaurant = (
    restaurant_data["name"].str.casefold()
    == "corner house ice cream"
)
similarity_scores[same_restaurant.to_numpy()] = -1

# Retrieve 100 similar candidates
similar_indices = similarity_scores.argsort()[::-1][:100]

candidates = restaurant_data.iloc[similar_indices].copy()
candidates["similarity"] = similarity_scores[similar_indices]
candidates = candidates.dropna(subset=["average_rating"])

# Normalize similarity within the candidate pool
max_similarity = candidates["similarity"].max()
candidates["similarity_normalized"] = (
    candidates["similarity"] / max_similarity
)

# Normalize restaurant ratings
candidates["rating_normalized"] = (
    candidates["average_rating"] / 5
)

# Calculate combined recommendation score
candidates["recommendation_score"] = (
    0.7 * candidates["similarity_normalized"]
    + 0.3 * candidates["rating_normalized"]
)

# Rank and remove repeated restaurant brands
candidates = candidates.sort_values(
    ["recommendation_score", "similarity"],
    ascending=[False, False]
)

candidates["normalized_name"] = (
    candidates["name"].str.casefold().str.strip()
)

candidates = candidates.drop_duplicates(
    subset=["normalized_name"]
)

print(
    candidates[
        ["name", "location", "average_rating",
         "similarity", "recommendation_score"]
    ].head(10).to_string(index=False)
)

                     name           location  average_rating  similarity  recommendation_score
             Gelato Pazzo                BTM             3.8    0.204200                 0.928
               Polar Bear          Bellandur             4.2    0.182712              0.878339
                Roll Over          Jayanagar             4.1    0.163044              0.804917
               Juice Shop                BTM          3.5375    0.166998               0.78472
   Robin Brown Ice Creams        Rajajinagar             3.5    0.161918              0.765057
        Iceberg Icecreams       Basavanagudi             4.0    0.144031              0.733738
                 Ice Land           JP Nagar             3.6    0.144826              0.712465
TFI - The Fresh Ice Cream  Bannerghatta Road             3.2    0.147469              0.697527
                 Ice Fort Basaveshwara Nagar             3.7    0.138428              0.696533
         Milano Ice Cream        Indiranagar      

## Integrate balanced ranking into our recommendation function

We will update the recommendation function to combine TF-IDF similarity and customer ratings.
The function will retain our existing improvements: location selection, exclusion of the searched restaurant's outlets, and unique restaurant recommendations.

In [74]:
def recommend_restaurants(
    restaurant_name,
    location=None,
    top_n=10,
    candidate_count=100,
    similarity_weight=0.7
):
    # Find the requested restaurant
    matches = restaurant_data[
        restaurant_data["name"].str.casefold()
        == restaurant_name.strip().casefold()
    ]

    if matches.empty:
        print("Restaurant not found. Please check the name.")
        return

    # Request a location when multiple outlets exist
    if location is None and len(matches) > 1:
        print("Multiple outlets found. Please specify a location:")
        print(matches["location"].to_string(index=False))
        return

    if location is not None:
        matches = matches[
            matches["location"].str.casefold()
            == location.strip().casefold()
        ]

        if matches.empty:
            print("Restaurant not found at the specified location.")
            return

    selected_index = matches.index[0]

    # Calculate review similarity
    similarity_scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix
    ).flatten()

    # Exclude all outlets of the searched restaurant
    same_restaurant = (
        restaurant_data["name"].str.casefold()
        == restaurant_data.loc[selected_index, "name"].casefold()
    )
    similarity_scores[same_restaurant.to_numpy()] = -1

    # Select the most similar candidates
    similar_indices = similarity_scores.argsort()[::-1][:candidate_count]

    recommendations = restaurant_data.iloc[similar_indices].copy()
    recommendations["similarity"] = similarity_scores[similar_indices]
    recommendations = recommendations.dropna(subset=["average_rating"])

    if recommendations.empty:
        print("No rated recommendations available.")
        return

    # Normalize similarity and rating
    max_similarity = recommendations["similarity"].max()

    recommendations["similarity_normalized"] = (
        recommendations["similarity"] / max_similarity
        if max_similarity > 0 else 0
    )

    recommendations["rating_normalized"] = (
        recommendations["average_rating"] / 5
    )

    # Calculate balanced recommendation score
    recommendations["recommendation_score"] = (
        similarity_weight * recommendations["similarity_normalized"]
        + (1 - similarity_weight) * recommendations["rating_normalized"]
    )

    # Rank by combined score
    recommendations = recommendations.sort_values(
        ["recommendation_score", "similarity"],
        ascending=[False, False]
    )

    # Keep one outlet per recommended restaurant
    recommendations["normalized_name"] = (
        recommendations["name"].str.casefold().str.strip()
    )

    recommendations = recommendations.drop_duplicates(
        subset=["normalized_name"]
    )

    return recommendations[
        ["name", "location", "average_rating",
         "max_votes", "similarity", "recommendation_score"]
    ].head(top_n).reset_index(drop=True)


# Test the updated recommendation function
recommend_restaurants(
    "Corner House Ice Cream",
    location="Banashankari",
    top_n=10
)

,name,location,average_rating,max_votes,similarity,recommendation_score
0,Gelato Pazzo,BTM,3.8,47,0.204200,0.928
1,Polar Bear,Bellandur,4.2,214,0.182712,0.878339
2,Roll Over,Jayanagar,4.1,439,0.163044,0.804917
3,Juice Shop,BTM,3.5375,29,0.166998,0.78472
4,Robin Brown Ice Creams,Rajajinagar,3.5,13,0.161918,0.765057
5,Iceberg Icecreams,Basavanagudi,4.0,43,0.144031,0.733738
6,Ice Land,JP Nagar,3.6,34,0.144826,0.712465
7,TFI - The Fresh Ice Cream,Bannerghatta Road,3.2,41,0.147469,0.697527
8,Ice Fort,Basaveshwara Nagar,3.7,23,0.138428,0.696533
9,Milano Ice Cream,Indiranagar,4.9,2090,0.112688,0.680296


## Evaluate the balanced model across different restaurant categories

We will test our updated recommendation function on three restaurant categories:
- KFC — Fast food
- Cafe Coffee Day — Coffee and cafés
- Faasos — Rolls and quick-service meals
The objective is to check whether the balanced ranking produces relevant and diverse recommendations across different food categories.

In [75]:
test_restaurants = [
    ("KFC", "Basavanagudi"),
    ("Cafe Coffee Day", "Banashankari"),
    ("Faasos", "Indiranagar")
]

for restaurant_name, location in test_restaurants:
    print(f"\n{'=' * 60}")
    print(f"Restaurant: {restaurant_name} | Location: {location}")
    print(f"{'=' * 60}")

    recommendations = recommend_restaurants(
        restaurant_name=restaurant_name,
        location=location,
        top_n=5
    )

    print(recommendations.to_string(index=False))


Restaurant: KFC | Location: Basavanagudi
             name              location  average_rating  max_votes  similarity  recommendation_score
Meat And Eat (ME)      Old Airport Road             3.6         97    0.177131                 0.916
      Big Chicken          Kammanahalli             3.9        118    0.170282              0.906933
     Meat And Eat              JP Nagar             3.7         81    0.170718              0.896656
         Truffles Koramangala 5th Block             4.7      14726    0.153694              0.889379
 BOX8- Desi Meals           Rajajinagar             4.1        559    0.159522              0.876412

Restaurant: Cafe Coffee Day | Location: Banashankari
      name              location  average_rating  max_votes  similarity  recommendation_score
   Barista Koramangala 5th Block             3.5         66    0.169810                  0.91
Cafe Mondo          Basavanagudi             3.7        514    0.164089              0.898416
     Cuppa      

## Compare different recommendation weights

Our current model assigns 70% importance to review similarity and 30% to ratings. These weights were chosen experimentally.
We will compare three configurations using Corner House Ice Cream:
- 50% similarity / 50% rating
- 70% similarity / 30% rating
- 90% similarity / 10% rating

This will help us understand how changing the weights affects recommendation relevance and ranking.

In [76]:
# Compare different similarity weights
weights = [0.5, 0.7, 0.9]

for weight in weights:
    print(f"\n{'=' * 60}")
    print(f"Similarity: {weight:.0%} | Rating: {1 - weight:.0%}")
    print(f"{'=' * 60}")

    recommendations = recommend_restaurants(
        restaurant_name="Corner House Ice Cream",
        location="Banashankari",
        top_n=5,
        similarity_weight=weight
    )

    print(
        recommendations[
            ["name", "average_rating", "similarity", "recommendation_score"]
        ].to_string(index=False)
    )


Similarity: 50% | Rating: 50%
            name  average_rating  similarity  recommendation_score
    Gelato Pazzo             3.8    0.204200                  0.88
      Polar Bear             4.2    0.182712              0.867385
       Roll Over             4.1    0.163044              0.809226
Milano Ice Cream             4.9    0.112688              0.765925
      Juice Shop          3.5375    0.166998              0.762657

Similarity: 70% | Rating: 30%
                  name  average_rating  similarity  recommendation_score
          Gelato Pazzo             3.8    0.204200                 0.928
            Polar Bear             4.2    0.182712              0.878339
             Roll Over             4.1    0.163044              0.804917
            Juice Shop          3.5375    0.166998               0.78472
Robin Brown Ice Creams             3.5    0.161918              0.765057

Similarity: 90% | Rating: 10%
                  name  average_rating  similarity  recommendation_

## Evaluate recommendation diversity across the model

We will evaluate 30 randomly selected restaurant outlets using our current 70:30 model.
We'll measure:
- Whether the model returns 10 recommendations.
- Whether the recommendations contain unique restaurant names.
- The average cosine similarity of recommended restaurants.

These are diagnostic metrics, not direct measures of recommendation accuracy.

In [77]:
# Select a reproducible sample of restaurant outlets
sample_restaurants = restaurant_data.sample(n=30, random_state=42)

evaluation_results = []

for _, restaurant in sample_restaurants.iterrows():
    recommendations = recommend_restaurants(
        restaurant_name=restaurant["name"],
        location=restaurant["location"],
        top_n=10,
        similarity_weight=0.7
    )

    if recommendations is not None:
        evaluation_results.append({
            "restaurant": restaurant["name"],
            "recommendation_count": len(recommendations),
            "unique_brands": recommendations["name"].str.casefold().nunique(),
            "average_similarity": recommendations["similarity"].mean()
        })

evaluation_df = pd.DataFrame(evaluation_results)

print("Restaurants evaluated:", len(evaluation_df))
print("Average recommendations:", round(evaluation_df["recommendation_count"].mean(), 2))
print("Average unique brands:", round(evaluation_df["unique_brands"].mean(), 2))
print("Average cosine similarity:", round(evaluation_df["average_similarity"].mean(), 4))
print("Restaurants with fewer than 10 recommendations:",
      (evaluation_df["recommendation_count"] < 10).sum())

Restaurants evaluated: 30
Average recommendations: 10.0
Average unique brands: 10.0
Average cosine similarity: 0.1447
Restaurants with fewer than 10 recommendations: 0


## Validate model and data consistency

We will perform final consistency checks before saving our model.
These checks will verify that:
- Every restaurant has a corresponding TF-IDF vector.
- Restaurant identities are unique.
- The TF-IDF matrix contains no invalid numerical values.
- The model has the expected number of features.

In [78]:
import numpy as np

# Validate model and restaurant data
print("Restaurant records:", len(restaurant_data))
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("TF-IDF vocabulary size:", len(tfidf_vectorizer.vocabulary_))

print("\nValidation checks:")

print(
    "Row alignment:",
    len(restaurant_data) == tfidf_matrix.shape[0]
)

print(
    "Unique restaurant outlets:",
    not restaurant_data.duplicated(
        subset=["name", "location"]
    ).any()
)

print(
    "Sequential DataFrame index:",
    restaurant_data.index.equals(
        pd.RangeIndex(len(restaurant_data))
    )
)

print(
    "Valid TF-IDF values:",
    np.isfinite(tfidf_matrix.data).all()
)

print(
    "Feature alignment:",
    tfidf_matrix.shape[1]
    == len(tfidf_vectorizer.vocabulary_)
)

Restaurant records: 5128
TF-IDF matrix shape: (5128, 50000)
TF-IDF vocabulary size: 50000

Validation checks:
Row alignment: True
Unique restaurant outlets: True
Sequential DataFrame index: True
Valid TF-IDF values: True
Feature alignment: True


## Save the trained model artifacts

We will save three artifacts using joblib:
1. tfidf_vectorizer.joblib — The trained TF-IDF vectorizer.
2. tfidf_matrix.joblib — The restaurant review vectors.
3. restaurant_data.joblib — Restaurant information, ratings, and reviews.

Saving these artifacts means our deployed application can load the existing model without retraining it every time the app starts.

In [79]:
import joblib
from pathlib import Path

# Create a directory for model artifacts
model_dir = Path("models")
model_dir.mkdir(exist_ok=True)

# Save trained model artifacts
joblib.dump(tfidf_vectorizer, model_dir / "tfidf_vectorizer.joblib")
joblib.dump(tfidf_matrix, model_dir / "tfidf_matrix.joblib")
joblib.dump(restaurant_data, model_dir / "restaurant_data.joblib")

# Verify saved files
for file_path in model_dir.glob("*.joblib"):
    print(
        f"{file_path.name}: "
        f"{file_path.stat().st_size / (1024 ** 2):.2f} MB"
    )

restaurant_data.joblib: 45.31 MB
tfidf_matrix.joblib: 29.95 MB
tfidf_vectorizer.joblib: 1.96 MB


## Load and verify saved model artifacts

We will reload the three saved artifacts and verify that they match our original model.

This confirms that the files were saved correctly and can be used for inference without retraining.

In [80]:
# Load saved model artifacts
loaded_vectorizer = joblib.load(model_dir / "tfidf_vectorizer.joblib")
loaded_matrix = joblib.load(model_dir / "tfidf_matrix.joblib")
loaded_restaurant_data = joblib.load(model_dir / "restaurant_data.joblib")

# Verify loaded artifacts
print("Loaded restaurant records:", len(loaded_restaurant_data))
print("Loaded TF-IDF matrix shape:", loaded_matrix.shape)
print("Loaded vocabulary size:", len(loaded_vectorizer.vocabulary_))

print("\nVerification checks:")

print(
    "Restaurant data matches:",
    loaded_restaurant_data.equals(restaurant_data)
)

print(
    "TF-IDF matrix matches:",
    (loaded_matrix != tfidf_matrix).nnz == 0
)

print(
    "Vocabulary matches:",
    loaded_vectorizer.vocabulary_ == tfidf_vectorizer.vocabulary_
)

Loaded restaurant records: 5128
Loaded TF-IDF matrix shape: (5128, 50000)
Loaded vocabulary size: 50000

Verification checks:
Restaurant data matches: True
TF-IDF matrix matches: True
Vocabulary matches: True


## Test recommendations using saved model artifacts

We will simulate how the deployed application will work by loading the saved model data into our recommendation function.

This is an inference test. The application should generate recommendations using the saved TF-IDF matrix and restaurant information without accessing the original CSV or retraining the model.

In [81]:
# Preserve the original notebook objects
original_restaurant_data = restaurant_data
original_tfidf_matrix = tfidf_matrix

# Use the reloaded artifacts for inference
restaurant_data = loaded_restaurant_data
tfidf_matrix = loaded_matrix

# Generate recommendations using saved artifacts
recommendations = recommend_restaurants(
    restaurant_name="Corner House Ice Cream",
    location="Banashankari",
    top_n=5
)

print(recommendations.to_string(index=False))

# Restore the original notebook objects
restaurant_data = original_restaurant_data
tfidf_matrix = original_tfidf_matrix

                  name    location  average_rating  max_votes  similarity  recommendation_score
          Gelato Pazzo         BTM             3.8         47    0.204200                 0.928
            Polar Bear   Bellandur             4.2        214    0.182712              0.878339
             Roll Over   Jayanagar             4.1        439    0.163044              0.804917
            Juice Shop         BTM          3.5375         29    0.166998               0.78472
Robin Brown Ice Creams Rajajinagar             3.5         13    0.161918              0.765057
